In [54]:
import os

import tensorflow  as tf
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential, layers, losses, metrics
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

In [46]:
X, y = fetch_california_housing(as_frame=True, return_X_y=True)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

from pyspark.resource import requestsfrom future.moves.urllib import response# Train and Compile Model

In [48]:
model = Sequential([
    layers.InputLayer(shape=([X.shape[1], ])),
    layers.Dense(128),
    layers.Dense(32),
    layers.Dense(1),
])

model.compile(
    optimizer='adam',
    loss=losses.Huber(),
    metrics=[metrics.MeanSquaredError(),
             metrics.R2Score(),
             metrics.RootMeanSquaredError()]
)

model.fit(X_train, y_train,
          epochs=10,
          batch_size=16,
          validation_data=(X_test, y_test),
          verbose=False
          )

C:\Users\username\Projects\MachineLearning\.venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


# Write Model using TensorFlow implementation

In [50]:
model_version = '0001'
model_name = 'models/fetch_california_housing'
model_path = os.path.join(model_name, model_version)
os.makedirs(model_path, exist_ok=True)
model_path = os.path.join(model_path, 'model.keras')
model.save(model_path)
# Alternative method
# tf.keras.models.save_model(model, model_path)

# Load Saved model

In [55]:
loaded_model = tf.keras.models.load_model(model_path)

X_sample = X_test.sample(10).astype('float16')
y_sample = y_test.loc[X_sample.index]

y_pred = loaded_model.predict(X_sample)

MSE = mean_squared_error(y_sample, y_pred)
MSE

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step


1.2565399724982735

# Deploy Model into Docker Container

1. Install Docker
2. Download image ```shell docker pull tensorflow/serving```
3. Build container ```shell docker run --it --rm -p 8500:8500 -p 8501:8501 -v "$ML_PATH/fetch_california_housing: /models/fetch_california_housing" -e MODEL_NAME=fetch_california_housing tensorflow/serving```

> -v "$ML_PATH/fetch_california_housing:/models/fetch_california_housing" - is data bridge , where  $ML_PATH/fetch_california_housing is local data folder
> -e MODEL_NAME=fetch_california_housing - decide which model need to serve ( automatically serve last known version )

# TF Serving + REST API communication

```python
import json
import numpy as np
import requests

X_new = np.ndarray([])
server_url = "http://localhost:8501/v1/models/fetch_california_housing:predict"

input_data_json = json.dumps({
    'signature_name': "serving_default",
    "instances": X_new.tolist()
})
headers = {"content-type": "application/json"}

response = requests.post(url=url, data=input_data_json, headers=headers)
response.raise_for_status()
response_json = response.json()

y_pred = np.array(response_json.get('predictions'))
y_pred.round(2)
```

# TF-Serving + gRPC

```python
import tensorflow as tf
from tensorflow_serving.apis.predict_pb2 import PredictRequest

request = PredictRequest()

request.model_spec.name = model_name
request.model_spec.signature_name = "serving_default"
input_name = model.input_names[0]

request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new))
````

```python
import grpc
from tensorflow_serving.apis import prediction_service_pb2_grpc

channel = grpc.insecure_channel('localhost:8500')
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel)
response = predict_service.Predict(request, timeout=10.0) # 10 sec timeout
```

```python
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
y_proba = tf.make_ndarray(outputs_proto)
```

## TF Lite ( convert )

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model('/path/to/model.keras') # Create converter
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE] # Decrease size of model
tf_lite_model = converter.convert() # transform model

with open('mobile_model.tflite', 'wb') as f: # Save it
    f.write(tf_lite_model)

## TF in Browser

```javascript
import * from '@tensorflow/tfjs'

const model = await tf.loadLayersModel('/path/to/model.json');
const image = tf.fromPixels(webcamElement);
const prediction = model.predict(image);
```

## GPU Boosting with TensorFlow

In [ ]:
# Check if GPU is available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

for gpu in tf.config.list_physical_devices('GPU'): # Set memory growth to prevent TensorFlow from allocating all GPU memory at once
    tf.config.experimental.set_memory_growth(gpu, True)

%run nvidia-smi # Check GPU usage

a = tf.Variable(42.0)
a.device # Check where variable is stored ( GPU or CPU )

with tf.device('/GPU:0'): # Force TensorFlow to use GPU
    b = tf.Variable(42.0)
    print(b.device)

## GPU Parallelism  Strategy
> - Data Parallelism - split data into batches and process them in parallel across multiple GPUs
> - Model Parallelism - split model into parts and process them in parallel across multiple GPUs ( vertical or horizontal )

In [ ]:
strategy = tf.distribute.MirroredStrategy() # Create strategy for data parallelism
with strategy.scope(): # Build and compile model within strategy scope
    model = Sequential([
        layers.InputLayer(shape=([X.shape[1], ])),
        layers.Dense(128),
        layers.Dense(32),
        layers.Dense(1),
    ])

    model.compile(
        optimizer='adam',
        loss=losses.Huber(),
        metrics=[metrics.MeanSquaredError(),
                 metrics.R2Score(),
                 metrics.RootMeanSquaredError()]
    )

batch_size = 100
model.fit(X_train, y_train,
          epochs=10,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          verbose=False
          )


distribution = tf.distribute.experimental.CentralStorageStrategy() # Create strategy for model parallelism
with distribution.scope(): # Build and compile model within strategy scope
    model = Sequential([
        layers.InputLayer(shape=([X.shape[1], ])),
        layers.Dense(128),
        layers.Dense(32),
        layers.Dense(1),
    ])

## TF Cluster Example

In [ ]:
cluster_spec = {
    "worker": ["worker0.example.com:2222", "worker1.example.com:2222"], # worker nodes for data parallelism
    "ps": ["ps0.example.com:2222"] # parameter server for model parallelism ( optional for data parallelism )
}

# on worker nodes

import os
import json

os.environ['TF_CONFIG'] = json.dumps({
    'cluster': cluster_spec,
    'task': {'type': 'worker', 'index': 0} # or index 1 for worker1
})